# Final local Exp1-Exp4 smoke test

This notebook uses the repository's existing vLLM endpoints, adapters, wrapper, rubrics, and experiment scripts. It creates no questions and no model outputs. The default run uses 10 real rows, exactly 2 from each clinical domain.

In [ ]:
# Runtime configuration
from pathlib import Path
import os, sys, json, time, subprocess, urllib.request
import pandas as pd

ROOT = Path.cwd()
DATASET = ROOT / 'benchmark_dataset' / '1000_questions_dataset.csv'
TEST_ROWS = 10
RUN_NAME = 'smoke10'
AUTO_LAUNCH_MODELS = True
HF_CACHE = Path(os.environ.get('HF_CACHE', '/lustre/smuexa01/client/users/mkotha/CS7325/hf_models'))
VLLM_PYTHON = os.environ.get('VLLM_PYTHON', sys.executable)
LOG_DIR = ROOT / 'results' / RUN_NAME / 'vllm_logs'
RUN_DIR = ROOT / 'results' / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)
JUDGE_CONFIGS = [
    {'id': 'medgemma', 'hf_id': 'google/medgemma-4b-it', 'port': 8001, 'gpu': '0', 'max_len': 4096, 'extra': '--trust-remote-code --enforce-eager'},
    {'id': 'biomistral', 'hf_id': 'BioMistral/BioMistral-7B', 'port': 8002, 'gpu': '1', 'max_len': 4096, 'extra': '--enforce-eager'},
    {'id': 'meditron', 'hf_id': 'epfl-llm/meditron-7b', 'port': 8003, 'gpu': '2', 'max_len': 2048, 'extra': '--enforce-eager'},
    {'id': 'medalpaca', 'hf_id': 'medalpaca/medalpaca-7b', 'port': 8004, 'gpu': '3', 'max_len': 2048, 'extra': '--enforce-eager'},
]
print('Repository:', ROOT)
print('Dataset:', DATASET)
print('Rows requested:', TEST_ROWS)

In [ ]:
# Validate and select only real source rows; no synthesis or rewriting
required = {'id', 'question', 'answer', 'domain', 'source_dataset', 'expected_class'}
assert DATASET.exists(), f'Missing dataset: {DATASET}'
all_rows = pd.read_csv(DATASET, dtype=str).fillna('')
assert required <= set(all_rows.columns), f'Missing columns: {sorted(required - set(all_rows.columns))}'
assert all_rows['id'].is_unique, 'Dataset IDs are not unique'
assert all_rows['question'].str.strip().ne('').all() and all_rows['answer'].str.strip().ne('').all(), 'Blank question or answer found'
domains = ['Cardiology', 'Pharmacology', 'Neurology', 'Pediatrics', 'Emergency']
assert TEST_ROWS % len(domains) == 0, 'TEST_ROWS must divide evenly across five domains'
per_domain = TEST_ROWS // len(domains)
test_df = pd.concat([all_rows[all_rows['domain'].eq(d)].head(per_domain) for d in domains], ignore_index=True)
assert len(test_df) == TEST_ROWS and test_df['domain'].value_counts().eq(per_domain).all(), 'Balanced real subset was not created'
test_df.to_csv(RUN_DIR / 'test_dataset.csv', index=False)
print(test_df[['id', 'domain', 'source_dataset']].to_string(index=False))
print('Real rows selected:', len(test_df))

In [ ]:
# Resolve model snapshots and optionally launch the four existing local models
def snapshot_for(hf_id):
    snap_root = HF_CACHE / ('models--' + hf_id.replace('/', '--')) / 'snapshots'
    snapshots = sorted([p for p in snap_root.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime) if snap_root.exists() else []
    if not snapshots: raise FileNotFoundError(f'{hf_id} not cached under {snap_root}')
    return snapshots[-1]

snapshots = {}
missing_models = []
for judge in JUDGE_CONFIGS:
    try:
        snapshots[judge['id']] = snapshot_for(judge['hf_id'])
        print(judge['id'], '->', snapshots[judge['id']])
    except FileNotFoundError as exc:
        missing_models.append(str(exc))
if AUTO_LAUNCH_MODELS and missing_models:
    raise RuntimeError('Cannot run a true model test; required local weights are missing:\n' + '\n'.join(missing_models))

processes = {}
if AUTO_LAUNCH_MODELS:
    for judge in JUDGE_CONFIGS:
        log_path = LOG_DIR / f"vllm_{judge['id']}.log"
        command = [VLLM_PYTHON, '-m', 'vllm.entrypoints.openai.api_server', '--model', str(snapshots[judge['id']]), '--port', str(judge['port']), '--host', '127.0.0.1', '--gpu-memory-utilization', '0.85', '--max-model-len', str(judge['max_len']), '--dtype', 'bfloat16'] + judge['extra'].split()
        log_handle = open(log_path, 'w')
        processes[judge['id']] = subprocess.Popen(command, stdout=log_handle, stderr=subprocess.STDOUT, env={**os.environ, 'CUDA_VISIBLE_DEVICES': judge['gpu'], 'HF_HUB_OFFLINE': '1', 'TRANSFORMERS_OFFLINE': '1', 'HF_HOME': str(HF_CACHE)})
        print('Started', judge['id'], 'PID', processes[judge['id']].pid)

def wait_for_models(timeout_seconds=1200):
    import httpx
    ready = {}
    started = time.time()
    while time.time() - started < timeout_seconds and len(ready) < len(JUDGE_CONFIGS):
        for judge in JUDGE_CONFIGS:
            if judge['id'] in ready: continue
            try:
                response = httpx.get(f"http://127.0.0.1:{judge['port']}/v1/models", timeout=3)
                if response.status_code == 200 and response.json().get('data'):
                    ready[judge['id']] = response.json()['data'][0]['id']
                    print('Ready:', judge['id'], ready[judge['id']])
            except Exception: pass
        if len(ready) < len(JUDGE_CONFIGS): time.sleep(10)
    if len(ready) != len(JUDGE_CONFIGS):
        failed = [x['id'] for x in JUDGE_CONFIGS if x['id'] not in ready]
        raise RuntimeError(f'Model startup failed or timed out: {failed}')
    return ready

served_models = wait_for_models()

In [ ]:
# Build temporary run configs without modifying repository configs
rubric_paths = [f'rubrics/rubric{i}.json' for i in ['1_pemat', '2_healthbench', '3_clinical_eval', '4_prometheus', '5_pemat_likert']]
judge_dicts = [{'id': j['id'], 'model': served_models[j['id']], 'url': f"http://127.0.0.1:{j['port']}"} for j in JUDGE_CONFIGS]
exp2_config = {
    'experiment': 'exp2_agreement_smoke10', 'judges': judge_dicts, 'rubrics': rubric_paths,
    'benchmark_csv': str(RUN_DIR / 'test_dataset.csv'), 'domains': domains, 'agreement_threshold': 80.0,
    'temperature': 0.0, 'max_tokens': 350, 'timeout_seconds': 300,
    'output_files': {'results_json': str(RUN_DIR / 'exp2_agreement_results.json')}
}
exp3_config = {**exp2_config, 'experiment': 'exp3_sensitivity_smoke10', 'max_questions': TEST_ROWS,
               'scoring_variants': ['BINARY', 'LIKERT_1_5', 'SCALED_0_10'],
               'output_files': {'results_json': str(RUN_DIR / 'exp3_sensitivity_results.json')}}
exp4_config = {**exp2_config, 'experiment': 'exp4_boxplots_smoke10', 'categories': domains,
               'output_files': {'results_json': str(RUN_DIR / 'exp4_boxplot_data.json'), 'figure_dir': str(RUN_DIR / 'figures')}}
exp1_config = json.loads((ROOT / 'config' / 'configs' / 'config_exp1_dataset.json').read_text())
exp1_config['output_files']['results_json'] = str(RUN_DIR / 'exp1_dataset_table.json')
exp1_input = test_df.rename(columns={'answer': 'reference_answer', 'source_dataset': 'source'})
exp1_input_path = RUN_DIR / 'exp1_dataset.csv'; exp1_input.to_csv(exp1_input_path, index=False)
config_paths = {'exp1': RUN_DIR / 'config_exp1.json', 'exp2': RUN_DIR / 'config_exp2.json', 'exp3': RUN_DIR / 'config_exp3.json', 'exp4': RUN_DIR / 'config_exp4.json'}
for key, value in [('exp1', exp1_config), ('exp2', exp2_config), ('exp3', exp3_config), ('exp4', exp4_config)]: config_paths[key].write_text(json.dumps(value, indent=2))
print('Temporary configs written under', RUN_DIR)

In [ ]:
# Exp1: use the existing dataset analysis script on the real 10-row subset
import importlib
os.environ['DATASET_PATH'] = str(exp1_input_path)
os.environ['EXP_CONFIG'] = str(config_paths['exp1'])
import experiments.exp1_dataset_analysis as exp1_mod
importlib.reload(exp1_mod); exp1_mod.main()
assert (RUN_DIR / 'exp1_dataset_table.json').exists()
print('Exp1 passed')

In [ ]:
# Exp2: existing four-judge agreement pipeline with all three retry stages
os.environ['DATASET_PATH'] = str(RUN_DIR / 'test_dataset.csv')
os.environ['EXP_CONFIG'] = str(config_paths['exp2'])
os.environ['MAX_QUESTIONS'] = '0'
import experiments.exp2_agreement_analysis as exp2_mod
importlib.reload(exp2_mod); exp2_mod.main()
exp2_path = RUN_DIR / 'exp2_agreement_results.json'
exp2_data = json.loads(exp2_path.read_text())
assert len(exp2_data) == 5
assert all(len(block['results']) == TEST_ROWS for block in exp2_data)
assert not any(r.get('agreement_class') == 'skipped' for b in exp2_data for r in b['results']), 'Skipped rows found'
print('Exp2 passed:', len(exp2_data), 'rubrics x', TEST_ROWS, 'questions')

In [ ]:
# Exp3: existing rubric sensitivity pipeline on the same real rows
os.environ['EXP_CONFIG'] = str(config_paths['exp3'])
import experiments.exp3_rubric_sensitivity as exp3_mod
importlib.reload(exp3_mod); exp3_mod.main()
exp3_data = json.loads((RUN_DIR / 'exp3_sensitivity_results.json').read_text())
assert len(exp3_data) == 5
assert all(len(block['variants']) == 3 for block in exp3_data)
print('Exp3 passed: 5 rubrics x 3 scoring variants')

In [ ]:
# Exp4: existing agreement visualization pipeline
os.environ['EXP_CONFIG'] = str(config_paths['exp4'])
os.environ['EXP2_RESULTS_PATH'] = str(RUN_DIR / 'exp2_agreement_results.json')
import experiments.exp4_boxplot_agreement as exp4_mod
importlib.reload(exp4_mod); exp4_mod.main()
figures = list((RUN_DIR / 'figures').glob('*.png'))
assert figures, 'No Exp4 figures were generated'
print('Exp4 passed:', len(figures), 'figures')

In [ ]:
# Final smoke-test report; report facts only
from collections import Counter
summary = []
for block in exp2_data:
    classes = Counter(r['agreement_class'] for r in block['results'])
    summary.append({'rubric': block['rubric_id'], 'rows': len(block['results']), **dict(classes)})
summary_df = pd.DataFrame(summary)
display(summary_df)
report = {'run': RUN_NAME, 'dataset_rows': TEST_ROWS, 'models': served_models, 'exp2': summary, 'figures': [str(x) for x in figures]}
(RUN_DIR / 'smoke_test_report.json').write_text(json.dumps(report, indent=2))
print('Report:', RUN_DIR / 'smoke_test_report.json')

In [ ]:
# Cleanup only processes started by this notebook
for process in processes.values():
    if process.poll() is None: process.terminate()
print('Started model processes terminated')